# Computing z-gains on your own data

A **z-gain** measures how much a subject's normative-model z-score changed between two
timepoints, corrected for how strongly z-scores at those two ages are expected to
correlate anyway. Without that correction, a subject who simply regresses toward the
mean looks like they changed.

$$\text{z-gain} = \frac{Z_2 - r\,Z_1}{\sqrt{1 - r^2}}$$

where $r$ is the expected correlation between z-scores at $age_1$ and $age_2$, read
from a **velocity matrix**.

This notebook runs end to end on synthetic data as soon as you open it. Work through
it once to see the shapes, then set `USE_DEMO_DATA = False` in Step 1 and point it at
your own files.

## What you need

**1. A table of z-scores**, CSV or pickle, in *long* format — one row per subject per
visit — with four columns (names are configurable):

| column | default name | requirement |
|---|---|---|
| subject id | `ID_subject` | repeated across that subject's visits |
| age | `age` | numeric. **Used as an integer index** into the velocity matrix, so ages are effectively rounded down to whole years |
| visit id | `ID_visit` | only used to order rows that share an age |
| measure | *you name it* | the **z-score**, not the raw volume or thickness |

Other columns (diagnosis, sex, site) ride along and are ignored.

```
ID_subject  age  ID_visit  Right-Hippocampus
  sub-0001   65         1              -1.88
  sub-0001   66         2              -1.70
  sub-0001   68         3              -1.86
  sub-0002   71         1               0.42
```

**2. A velocity matrix** — an age-by-age correlation matrix, as a pickle holding either
the array itself or a dict with an `A_sparse_predict` key. You get this from fitting the
velocity model to your normative model. If you don't have one yet, the demo data below
generates a stand-in so you can still see the pipeline work.

## Setup

Install the package once, from the directory containing `zgain/`:

```bash
pip install -e zgain
```

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd

# If you ran `pip install -e zgain` this does nothing. Otherwise it looks for the
# package next to the notebook, so you can run without installing.
if "z_gain" not in sys.modules:
    try:
        import z_gain  # noqa: F401
    except ImportError:
        for _base in (Path.cwd(), *Path.cwd().parents[:3]):
            _src = _base / "zgain" / "src"
            if (_src / "z_gain" / "__init__.py").exists():
                sys.path.insert(0, str(_src))
                break
        else:
            raise ImportError(
                "Could not find the z_gain package.\n"
                "Run `pip install -e zgain` from the directory containing zgain/,\n"
                f"or start Jupyter from there. Current directory: {Path.cwd()}"
            )

from z_gain import (
    build_zgain_frame,
    compute_all_z_gains,
    example_frame,
    example_velocity_matrix,
    load_input_data,
    load_velocity_matrix,
    validate_zgain_frame,
)

print("z_gain imported from:", sys.modules["z_gain"].__file__)

## Step 1 — Configure

Set `USE_DEMO_DATA = False` and fill in the paths when you are ready to use your own data.

In [ ]:
USE_DEMO_DATA = True          # <-- set to False once the paths below are filled in

DATA_PATH = "path/to/your_z_scores.csv"       # .csv or .pkl
VELOCITY_PATH = "path/to/velocity_objects.pkl"  # a .pkl file, or a directory of
                                                # <measure>/velocity_objects.pkl
MEASURE = "Right-Hippocampus"                 # the column to analyse
OUTPUT_DIR = "z_gain_results"                 # where to write; None = don't write

# Your column names, if they differ from the defaults.
ID_COL = "ID_subject"
AGE_COL = "age"
VISIT_COL = "ID_visit"

COLS = dict(id_col=ID_COL, age_col=AGE_COL, visit_col=VISIT_COL)

## Step 2 — Load your data

If your data needs assembling first — joining model output to a covariate table,
merging in diagnosis labels, renaming columns — do it in this cell. Everything
downstream only cares about the four required columns.

In [ ]:
if USE_DEMO_DATA:
    MEASURE = "example_measure"
    df = example_frame(measure=MEASURE, **COLS)
    print("Using synthetic demo data.")
else:
    df = load_input_data(DATA_PATH)
    print(f"Loaded {DATA_PATH}")

print(f"{len(df)} rows, {df[ID_COL].nunique()} subjects")
df.head()

Check that the four required columns are present and sanely typed:

In [ ]:
missing = [c for c in (ID_COL, AGE_COL, VISIT_COL, MEASURE) if c not in df.columns]
if missing:
    raise KeyError(f"Missing {missing}. Your columns are: {list(df.columns)}")

print(df[[ID_COL, AGE_COL, VISIT_COL, MEASURE]].dtypes.to_string())
print(f"\nage range: {df[AGE_COL].min()} to {df[AGE_COL].max()}")
print(f"visits per subject: {df.groupby(ID_COL).size().describe()[['min','50%','max']].to_dict()}")

## Step 3 — Load the velocity matrix

The matrix is indexed by **integer age**, so it must cover your age range. Sparse
matrices and `{'A_sparse_predict': ...}` dicts are unwrapped automatically.

In [ ]:
if USE_DEMO_DATA:
    R = example_velocity_matrix()
    print("Using a synthetic AR(1) stand-in matrix.")
else:
    vpath = Path(VELOCITY_PATH)
    if vpath.is_dir():                      # a directory of per-measure matrices
        vpath = vpath / MEASURE / "velocity_objects.pkl"
    R = load_velocity_matrix(vpath)
    print(f"Loaded {vpath}")

print(f"shape {R.shape}, so ages 0 to {min(R.shape) - 1} are covered")
print(f"example: correlation between ages 65 and 70 is r = {R[70, 65]:.3f}")

## Step 4 — Validate before computing

**This step matters.** The computation catches per-pair errors and logs a warning, so a
malformed frame gives you a *smaller* result rather than an error. Pairs are dropped
when the age is NaN or outside the matrix, or when $r < 0.3$. NaN z-scores are worse:
they raise nothing and flow through as NaN gains.

`validate_zgain_frame` tells you all of that up front. Read the warnings — a result
with silently missing rows still looks perfectly fine.

In [ ]:
frame = build_zgain_frame(df, None, MEASURE, validate=False, **COLS)
report = validate_zgain_frame(frame, MEASURE, R=R, **COLS)
print(report.summary())

Two patterns worth understanding if you see them:

- **"N row(s) repeat an age within a subject"** — your ages collapse to the same whole
  year. Only the first (by visit order) survives. If your visits are months apart, this
  can discard a large fraction of your data.
- **"N pair(s) have r < 0.3"** — those two ages are too far apart for the velocity
  matrix to relate them. Often the matrix is banded, with `r` exactly 0 beyond some gap.

## Step 5 — Compute the z-gains

In [ ]:
gains = compute_all_z_gains(frame, R, MEASURE, **COLS)
print(f"{len(gains)} z-gains from {gains['id'].nunique()} subjects")
gains.head()

One row per **pair of timepoints**, not per visit:

| column | meaning |
|---|---|
| `id` | subject |
| `age1`, `age2` | the two ages, always `age2 > age1` |
| `t1_index`, `t2_index` | positions within that subject's visit sequence |
| `z_gain` | the corrected change |
| `Z1`, `Z2` | the input z-scores |

A subject with 3 distinct ages gives 3 rows; a subject with one visit gives none.

In [ ]:
print(gains["z_gain"].describe().to_string())
print(f"\nNaN z_gain rows: {int(gains['z_gain'].isna().sum())}  (should be 0)")

## Step 6 — Save

In [ ]:
if OUTPUT_DIR:
    out = Path(OUTPUT_DIR)
    out.mkdir(parents=True, exist_ok=True)
    dest = out / f"z_gains_{MEASURE}.pkl"
    gains.to_pickle(dest)
    print(f"Wrote {dest}")
else:
    print("OUTPUT_DIR is None, nothing written.")

## All measures at once

Once one measure works, loop. Note the velocity matrix is reloaded per measure if
`VELOCITY_PATH` is a directory, since each measure has its own.

In [ ]:
def run_all(df, velocity_path, measures, output_dir=None, **cols):
    """Compute z-gains for each measure. Returns {measure: gains DataFrame}."""
    results = {}
    for measure in measures:
        try:
            if USE_DEMO_DATA:
                R_m = example_velocity_matrix()
            else:
                vp = Path(velocity_path)
                R_m = load_velocity_matrix(
                    vp / measure / "velocity_objects.pkl" if vp.is_dir() else vp)

            frame_m = build_zgain_frame(df, None, measure, validate=False, **cols)
            rep = validate_zgain_frame(frame_m, measure, R=R_m, **cols)
            if not rep.ok:
                print(f"{measure}: SKIPPED\n  " + rep.summary().replace("\n", "\n  "))
                continue

            g = compute_all_z_gains(frame_m, R_m, measure, **cols)
            results[measure] = g

            if output_dir:
                Path(output_dir).mkdir(parents=True, exist_ok=True)
                g.to_pickle(Path(output_dir) / f"z_gains_{measure}.pkl")
            print(f"{measure}: {len(g)} gains")
        except Exception as exc:
            print(f"{measure}: FAILED, {type(exc).__name__}: {exc}")
    return results


# Every numeric column that is not an id, age or visit column.
measures = [c for c in df.columns
            if c not in {ID_COL, AGE_COL, VISIT_COL} and pd.api.types.is_numeric_dtype(df[c])]
print(f"{len(measures)} measure(s): {measures}\n")

all_gains = run_all(df, VELOCITY_PATH, measures, OUTPUT_DIR, **COLS)

## Next steps

- Join your grouping variables (diagnosis, sex, site) back onto the output on `id`, or
  pass them via `build_zgain_frame(..., keep_cols=[...])` so they ride along from the start.
- Compare `z_gain` distributions between groups, or test whether they differ from 0.
- `run_velocity_template.py` next to this notebook does the same thing as a command-line
  script, if you'd rather batch it than step through cells.